# OpenSynCity — Main 과정 한눈에 보기 (정전 시나리오, 완전 self-contained)

이 노트북 **하나만** 위에서 아래로 실행하면 우리 모델의 전체 동작을 이해·재현할 수 있습니다. 외부 모듈 import 없이 핵심 코드를 전부 인라인했고, 각 줄에 주석을 달았습니다.

---

## 큰 그림

### 1. 문제 설정 (무엇을 푸는가)

대상은 **여러 건물로 이루어진 도시 에너지 커뮤니티**입니다 (시뮬레이터: CityLearn 2022, `citylearn_challenge_2022_phase_all`). 각 건물 $b$ 는

- **비가변 부하** $L_b(t)$ (`non_shiftable_load`, kWh) — 끌 수 없는 소비,
- **태양광 발전** $S_b(t)$ (`solar_generation`, kWh),
- **배터리(ESS)** — 용량 $C_b$ (kWh)·최대 출력 $P_b$ (kW)·충전상태 $\mathrm{SoC}_b(t)\in[0,1]$

를 가지며, 커뮤니티 전체가 **요금** $\pi(t)$ (`electricity_pricing`, \$/kWh)와 **탄소강도** $g(t)$ (`carbon_intensity`, kgCO₂/kWh)를 공유합니다.

각 step(=1시간)마다 우리가 **결정하는 유일한 제어 변수**는 건물별 배터리 행동
$$a_b(t)\in[-1,+1]\qquad(-1:\text{최대 방전},\ +1:\text{최대 충전})$$
하나입니다. 건물의 **순부하(net load)** 는
$$\text{net}_b(t)=\underbrace{L_b(t)-S_b(t)}_{\text{기저 순부하}}+\,a_b(t)\,P_b$$
이고, 양수면 그리드에서 사오고(import) 음수면 흘려보냅니다. 

**목표는 두 가지를 동시에** 만족하는 것입니다.

1. **평상시 효율** — 전기요금·탄소배출·그리드 부담(ramping)을 낮춘다.
2. **돌발 회복력(resilience)** — 정전(power outage)으로 그리드가 끊겨 건물이 *섬(island)* 이 되면, 미리 비축해 둔 배터리로 부하를 버텨 **미공급 에너지(unserved energy)** 를 최소화한다.

이 둘은 **상충**합니다: 요금만 보면 배터리를 자주 비우는 게 이득이지만, 그러면 정전 순간 버틸 에너지가 없습니다. 우리 모델은 이 trade-off를 **상황에 따라 자동 전환**합니다.

### 2. 핵심 아이디어 — 에이전틱 메시 (Agentic Mesh)

중앙 컨트롤러 하나가 모든 건물을 지휘하는 대신, **여러 자율 에이전트가 공유 블랙보드(blackboard)로 소통**하며 협력합니다. 등장하는 에이전트는:

| 에이전트 | 역할 | 코드 |
|---|---|---|
| **BuildingAgent** (건물당 1개) | 자기 상태·예측을 블랙보드에 게시하고, 자기 배터리 MPC(LP)를 직접 푼다 | §4 |
| **SharedVarAgent** (요금·탄소) | 구역 공통 신호를 예측해 모두에게 broadcast | §4 |
| **RoleAssigner** | 매 step 건물 상태로 역할을 동적 라벨링(설명가능성) | §4 |
| **OutageRiskAgent** | 정전 빈도를 학습해 *선택적으로* 예비(reserve)를 켠다 | §5 |
| **MASMPCAgent** | 위 모두를 한 step 안에서 오케스트레이션 | §6 |

이 구조의 장점: **분산성**(각 건물이 자기 문제만 풀어 확장 용이)·**설명가능성**(에이전트별 자연어 근거 생성)·**모듈성**(정전 대응을 끄면 정확히 평상시 MPC로 환원).

### 3. 두뇌 — 모델예측제어 (MPC, Receding Horizon)

각 BuildingAgent는 매 step **앞으로 $H$시간**의 배터리 스케줄을 **선형계획(LP)** 으로 최적화하고, **첫 행동 $a_b(0)$만 실행**한 뒤 다음 step에 다시 풉니다(receding horizon). 예측은 **과거에 관측한 같은 시각(hour-of-day)의 평균**만 쓰므로 미래를 훔쳐보지 않습니다(*causal*).

**결정 변수** ($t=0,\dots,H-1$): 배터리 행동 $a_t$, 그리드 수입 $\text{imp}_t=\max(\text{net}_t,0)$, 시간 간 변동 $\text{ramp}_t=|\text{net}_t-\text{net}_{t-1}|$, (정전 대비) 예비 부족분 $\text{short}_t$.

**목적함수 (최소화):**
$$
\min\;\sum_{t=0}^{H-1}\big(\pi_t+\lambda_{\text{carbon}}\,g_t\big)\,\text{imp}_t
\;+\;\lambda_{\text{ramp}}\sum_{t}\text{ramp}_t
\;+\;\rho_{\text{reserve}}\sum_{t}\text{short}_t
$$

**제약:** 배터리 SoC 동역학 $\mathrm{SoC}_{t}=\mathrm{SoC}_0+\sum_{k\le t} a_k\cdot r$ 가 $[\mathrm{SoC}_{\min},\mathrm{SoC}_{\max}]$ 유지 (여기서 $r=P_b/C_b$ 는 *행동 1단위가 바꾸는 SoC 비율*), $\max(\cdot)$·$|\cdot|$ 를 부등식으로 선형화, 그리고 예비 하한 제약 $\text{short}_t\ge \text{reserve\_floor}-\mathrm{SoC}_t$.

> 핵심: $\rho_{\text{reserve}}$ 와 $\text{reserve\_floor}$ 는 **OutageRiskAgent가 정전 위험을 감지한 시간대에만** 켜집니다. 위험이 0이면 위 식의 마지막 항이 사라져 **순수 비용·탄소 최적화(=평상시 MPC)** 가 됩니다.

### 4. 세 단계(Phase)로 보는 동작

| 단계 | 하는 일 | 핵심 파라미터 | 이 노트북 섹션 |
|---|---|---|---|
| **Phase 1 — 분산 최적화** | 각 건물 에이전트가 예측을 공유하고 자기 배터리 MPC(LP)를 풀어 행동 결정. 평상시엔 중앙 MPC와 수치 동일 | `horizon`, `carbon_weight`, `ramp_weight` | 1~3, 6 |
| **Phase 2 — 돌발 대응** | 정전 위험을 *선택적으로* 학습 → 미리 배터리 비축(reserve) + 정전 순간 긴급 방전 | `risk_threshold`, `max_reserve`, `anticipate`, `reserve_penalty` | 5, 6 |
| **Phase 3 — 역할 분배** | 건물 상태(SoC·순부하)에 따라 역할을 동적으로 부여(설명가능성) | `soc_high`, `soc_low` | 4 |

### 5. 매개변수 사전 (각 파라미터의 의미)

**MPC / 배터리 (`MPCBattery`, `MPCConfig`)**

| 기호·이름 | 의미 | 직관 (값을 키우면) |
|---|---|---|
| `horizon` $H$ | 예측 지평 (기본 8시간) | 더 멀리 내다봄 — 선축적은 좋아지나 계산량↑·예측오차 누적 |
| `carbon_weight` $\lambda_{\text{carbon}}$ | 목적함수에서 탄소 대 요금의 상대 비중 (기본 1.0) | 탄소 적은 시간대로 충전 이동 — 친환경 ↔ 요금 |
| `ramp_weight` $\lambda_{\text{ramp}}$ | 시간 간 전력 변동(ramping) 패널티 (기본 0.3) | 그리드에 더 매끄러운 부하 — 첨두 완화 ↔ 비용 최적성 |
| `wear_weight` | 배터리 충·방전 마모 비중 (기본 0.001, 현재 미사용) | 사이클 억제 — 수명 ↔ 활용도 |
| `soc_min`, `soc_max` | SoC 허용 하·상한 (기본 0,1) | 안전 운전 구간 |
| `capacity` $C_b$, `nominal_power` $P_b$ | 건물 배터리 용량(kWh)·최대 출력(kW) | 환경이 정하는 물리 스펙 |
| `ratio` $r=P_b/C_b$ | 행동 1단위($a=1$)가 한 step에 바꾸는 SoC 비율 | 큰 출력/작은 용량일수록 SoC가 빨리 변함 |

**정전 대응 (`OutageRiskAgent`)**

| 이름 | 의미 | 직관 |
|---|---|---|
| `risk_threshold` | '위험 시간대' 판단 임계값 = *최고 정전빈도의 몇 배 이상인가* (기본 0.5) | 낮추면 더 많은 시간을 위험으로 봄(reserve 자주 켜짐), 높이면 저녁 피크만 |
| `max_reserve` | 위험 시 강제하는 SoC 예비 하한의 최대치 (SoC 비율, 기본 0.4) | 정전 대비 비축량 — 회복력 ↔ 평상 효율 |
| `anticipate` | 위험 시간대 *몇 시간 전부터* 미리 비축할지 (기본 3) | 일찍 채워 안전 ↔ 그 시간 효율 손해 |
| `reserve_penalty` $\rho_{\text{reserve}}$ | SoC가 예비 하한 미달일 때 LP가 받는 패널티 (기본 20.0) | 클수록 reserve를 '거의 hard 제약'처럼 지킴 |

**역할 배정 (`RoleAssigner`)** — 행동은 안 바꾸고 라벨만 부여: `soc_high`(기본 0.6)·`soc_low`(기본 0.3) 기준으로 `relief_capable`(여유 방전 가능)·`absorber`(잉여 흡수)·`reserve_holder`(예비 보존)·`flexible`(유연).

**정전 시나리오 주입 (`enable_outages`)** — 신뢰도 지표 기반: `saifi`(연간 정전 *횟수*, System Average Interruption Frequency Index)·`caidi`(평균 *지속시간*, 분, Customer Average Interruption Duration Index)·`start_hours`(정전이 시작될 수 있는 시각, 저녁 피크 17~20시)·`seed`(모든 컨트롤러에 **같은 시드** → 동일 정전 → 공정 비교).

### 6. 무엇을 보일 것인가

마지막에 **무제어 / 규칙기반(RBC) / 강화학습(SAC) / MPC / 우리 모델(Mesh)** 을 같은 정전 시나리오에서 비교합니다. 평가 축은 (1) **표준 점수**(요금·탄소·ramping·load-factor의 평균, ↓ 좋음)와 (2) **정전 회복력**(`power_outage_normalized_unserved_energy_total`, 미공급 에너지, ↓ 좋음)입니다. 우리 모델은 평상시 효율을 MPC 수준으로 유지하면서 회복력에서 다른 방법들을 앞서는 것이 목표입니다.

## 0. 설치 + 경로 설정

외부 의존성은 **CityLearn(시뮬레이터) 하나뿐**입니다. CityLearn은 numpy 1.x에서 동작하므로, numpy 2.x면 다운그레이드 후 런타임을 재시작합니다(가드).

In [ ]:
# --- 실행 환경 감지: Colab이면 구글 드라이브 마운트, 아니면 로컬 경로 사용 ---
try:
    from google.colab import drive          # Colab 전용 모듈 (로컬엔 없음)
    drive.mount('/content/drive')           # 내 구글 드라이브를 /content/drive 에 연결
    from pathlib import Path
    ROOT = Path('/content/drive/MyDrive')    # 드라이브 최상위를 기준 경로로
except Exception:                            # Colab이 아니면(로컬 실행) 여기로
    from pathlib import Path
    ROOT = Path('/Users/seukoh/Desktop/Final_mesh1')   # 로컬 기준 경로 (본인 환경에 맞게 수정)

import sys
CITYLEARN  = ROOT/'CityLearn-master'                                  # CityLearn 시뮬레이터 폴더
SAC_BUNDLE = ROOT/'sacrbc'/'checkpoints'/'best_inference_bundle.pt'    # 학습된 강화학습 가중치(있으면 비교에 포함)
assert (CITYLEARN/'citylearn'/'citylearn.py').exists(), f'CityLearn-master 없음: {CITYLEARN}'  # 경로 확인

# --- numpy 버전 가드: CityLearn은 numpy<2 필요. 2.x면 다운그레이드 후 재시작 ---
import numpy as _np
if _np.__version__.startswith('2'):
    %pip install -q 'numpy<2.0.0' 'pandas<2.2' 'scipy<1.13' 'gymnasium<=0.28.1' platformdirs requests pyyaml simplejson 'xgboost>=1.7,<3' scikit-learn torch tqdm
    print('numpy<2 설치 완료 — 런타임을 재시작한 뒤 이 셀부터 다시 실행하세요.')
    import os; os.kill(os.getpid(), 9)       # 런타임 강제 재시작(설치 반영)

# --- CityLearn을 import 경로에 추가하고, 공통 라이브러리 로드 ---
sys.path.insert(0, str(CITYLEARN))
import citylearn, numpy as np, pandas as pd
from scipy.optimize import linprog           # 선형계획(LP) 솔버 — MPC 최적화에 사용
from dataclasses import dataclass            # 설정 객체용
print('준비 완료 | CityLearn', citylearn.__version__, '| numpy', _np.__version__)

## 1. 예측기 (Forecaster)

MPC는 "앞으로 H시간"을 내다보고 최적화합니다. 그러려면 앞으로의 **순부하(부하−태양광)·전기요금·탄소배출**을 예측해야 합니다.

여기서는 **과거에 관측한 같은 시각(hour-of-day)의 평균**으로 예측합니다 — 미래 데이터를 절대 쓰지 않으므로(causal) 정직합니다. (전기요금은 환경이 주는 1~3시간 예측을 먼저 쓰고, 그 다음은 시간대 평균.)

In [ ]:
class ForecasterMPC:
    """건물별 순부하/요금/탄소를 '시간대 평균'으로 예측 (과거 정보만 사용 → 미래 누설 없음)."""
    def __init__(self, n_buildings, horizon):
        self.n = n_buildings                 # 건물 수
        self.H = horizon                     # 예측 지평(시간)
        # 건물별·시각별(0~23시) 관측 기록 보관함 (이걸 평균내서 예측)
        self.net_hod    = [[[] for _ in range(24)] for _ in range(n_buildings)]  # 순부하 기록
        self.price_hod  = [[] for _ in range(24)]   # 요금 기록 (모든 건물 공통)
        self.carbon_hod = [[] for _ in range(24)]   # 탄소 기록 (공통)
        self.last_net   = np.zeros(n_buildings)     # 직전 순부하 (당장 다음 시각 예측의 기본값)
        self.last_price = 0.0; self.last_carbon = 0.0
        self.price_pred = []                 # 환경이 제공하는 요금 예측(1~3시간)

    def update(self, hour, load, solar, price, carbon, price_pred):
        """매 step 호출: 이번 관측을 기록함에 누적."""
        h = int(hour) % 24                                   # 현재 시각(0~23)
        net = np.asarray(load, dtype=float) - np.asarray(solar, dtype=float)  # 순부하 = 부하 - 태양광
        for b in range(self.n):
            self.net_hod[b][h].append(float(net[b]))         # 이 건물·이 시각 순부하 기록
            if len(self.net_hod[b][h]) > 60:                 # 최근 60일치만 유지(메모리·적응)
                self.net_hod[b][h] = self.net_hod[b][h][-60:]
        self.price_hod[h].append(float(price))               # 요금 기록
        self.carbon_hod[h].append(float(carbon))             # 탄소 기록
        if len(self.price_hod[h]) > 60:
            self.price_hod[h]  = self.price_hod[h][-60:]
            self.carbon_hod[h] = self.carbon_hod[h][-60:]
        self.last_net = net; self.last_price = float(price); self.last_carbon = float(carbon)
        self.price_pred = [float(x) for x in price_pred]      # 환경 제공 예측 저장

    def net_forecast(self, b, hour):
        """건물 b의 앞으로 H시간 순부하 예측 벡터."""
        out = [float(self.last_net[b])]                       # t=0(지금)은 직전 실측값
        for k in range(1, self.H):                            # t=1..H-1
            vals = self.net_hod[b][(int(hour)+k) % 24]        # k시간 뒤 시각의 과거 기록
            out.append(float(np.mean(vals)) if vals else float(self.last_net[b]))  # 평균(없으면 직전값)
        return np.asarray(out)

    def price_forecast(self, hour):
        """앞으로 H시간 요금 예측. 1~3시간은 환경 제공 예측, 그 뒤는 시간대 평균."""
        out = [self.last_price]
        for k in range(1, self.H):
            if k <= len(self.price_pred):                     # 환경이 준 예측이 있으면 그걸 사용
                out.append(self.price_pred[k-1])
            else:
                vals = self.price_hod[(int(hour)+k) % 24]
                out.append(float(np.mean(vals)) if vals else self.last_price)
        return np.asarray(out)

    def carbon_forecast(self, hour):
        """앞으로 H시간 탄소 예측 (시간대 평균)."""
        out = [self.last_carbon]
        for k in range(1, self.H):
            vals = self.carbon_hod[(int(hour)+k) % 24]
            out.append(float(np.mean(vals)) if vals else self.last_carbon)
        return np.asarray(out)

## 2. 건물별 배터리 MPC — 선형계획(LP)

이 부분이 모델의 "두뇌"입니다. 매 step, 한 건물의 배터리를 **앞으로 H시간** 어떻게 충·방전할지 **선형계획(LP)** 으로 최적화합니다.

**결정 변수** (t = 0 ~ H-1):
- `a_t` : 배터리 행동, −1(완전 방전) ~ +1(완전 충전)
- `imp_t` : 그리드에서 사오는 전력 = max(순전력, 0)
- `ramp_t` : 시간 간 전력 변동량 = |net_t − net_{t-1}|  (그리드 평활용)
- (정전 대비) `short_t` : SoC가 reserve 하한보다 부족한 양

**목적함수(최소화)**: `Σ (요금+탄소)·imp_t  +  램핑가중치·Σ ramp_t  +  reserve패널티·Σ short_t`

**제약**: 배터리 용량/출력 한계, SoC가 0~1 유지, max()·abs()를 부등식으로 선형화.

푼 뒤 **첫 행동 `a_0`만** 실행하고, 다음 step에 다시 푼다 = receding horizon(MPC).

In [ ]:
class MPCBattery:
    def __init__(self, capacity, nominal_power, horizon=8, soc_min=0.0, soc_max=1.0,
                 carbon_weight=1.0, ramp_weight=0.3, wear_weight=0.001, reserve_penalty=20.0):
        self.capacity = float(capacity)          # 배터리 용량 (kWh)
        self.nominal  = float(nominal_power)     # 배터리 최대 충·방전 출력 (kW)
        self.H = int(horizon)                    # 예측 지평
        self.soc_min = soc_min; self.soc_max = soc_max   # SoC 허용 범위 (0~1)
        self.carbon_weight = carbon_weight       # 목적함수에서 탄소 비중
        self.ramp_weight   = ramp_weight         # 램핑(그리드 평활) 비중
        self.wear_weight   = wear_weight         # 배터리 마모 비중(현재 미사용)
        self.reserve_penalty = reserve_penalty   # SoC가 reserve 하한 미달 시 패널티(정전 대비)
        self.ratio = self.nominal / max(self.capacity, 1e-6)   # 행동 1단위가 바꾸는 SoC 비율

    def decide(self, soc, base_net, price, carbon, coupling=None, reserve_floor=0.0):
        """편의 함수: LP를 풀어 첫 행동 a_0만 반환."""
        actions, _ = self.solve(soc, base_net, price, carbon, coupling, reserve_floor)
        return float(actions[0])

    def solve(self, soc, base_net, price, carbon, coupling=None, reserve_floor=0.0):
        """H시간 LP를 풀어 (행동열[H], 순전력열[H]) 반환."""
        H = self.H
        na, nimp, nramp = H, H, H-1                 # 변수 개수: 행동 H개, 수입 H개, 램핑 H-1개
        use_reserve = reserve_floor > 1e-9          # reserve 하한이 있으면 short 변수 추가
        nshort = H if use_reserve else 0
        N = na + nimp + nramp + nshort              # 전체 변수 개수
        ia, iimp, iramp = 0, na, na+nimp            # 각 변수 블록의 시작 인덱스
        ishort = na + nimp + nramp
        pi = np.zeros(H) if coupling is None else np.asarray(coupling, dtype=float)  # 피더 조율용(여기선 0)

        # ----- 목적함수 계수 c (min cᵀx) -----
        c = np.zeros(N)
        for t in range(H):
            c[iimp + t] = float(price[t]) + self.carbon_weight*float(carbon[t])  # 수입에 요금+탄소
            c[ia   + t] = float(pi[t]) * self.nominal                            # 피더 조율 항(0이면 무시)
        for t in range(nramp):
            c[iramp + t] = self.ramp_weight                                       # 램핑 패널티
        for t in range(nshort):
            c[ishort + t] = self.reserve_penalty                                  # reserve 미달 패널티

        # ----- 부등식 제약 A x <= b 를 한 줄씩 쌓는 헬퍼 -----
        A, b = [], []
        def row(coeffs, rhs):
            r = np.zeros(N)
            for k, v in coeffs.items(): r[k] = v
            A.append(r); b.append(rhs)

        # (1) imp_t >= net_t  (수입은 순전력 이상)  ->  a_t·nominal - imp_t <= -base_net_t
        for t in range(H):
            row({ia+t: self.nominal, iimp+t: -1.0}, -float(base_net[t]))
        # (2) ramp_t >= |net_t - net_{t-1}|  를 두 부등식으로 선형화
        for t in range(1, H):
            dbase = float(base_net[t] - base_net[t-1])
            row({ia+t: self.nominal, ia+(t-1): -self.nominal, iramp+(t-1): -1.0}, -dbase)
            row({ia+t: -self.nominal, ia+(t-1): self.nominal, iramp+(t-1): -1.0},  dbase)
        # (3) SoC 범위: soc + 누적(행동)·ratio 가 [soc_min, soc_max] 안에
        for t in range(H):
            row({ia+k:  self.ratio for k in range(t+1)}, self.soc_max - soc)   # 위로 soc_max 이하
            row({ia+k: -self.ratio for k in range(t+1)}, soc - self.soc_min)   # 아래로 soc_min 이상
        # (4) reserve(soft): short_t >= reserve_floor - SoC_t  (SoC가 하한보다 부족한 양)
        if use_reserve:
            for t in range(H):
                r = {ia+k: -self.ratio for k in range(t+1)}; r[ishort+t] = -1.0
                row(r, soc - reserve_floor)

        # ----- 변수 경계: 행동 [-1,1], 나머지(수입·램핑·short)는 0 이상 -----
        bounds = [(-1.0,1.0)]*na + [(0.0,None)]*nimp + [(0.0,None)]*nramp + [(0.0,None)]*nshort

        # ----- LP 풀기 (HiGHS 솔버, 이 작은 문제엔 매우 빠름) -----
        try:
            res = linprog(c, A_ub=np.asarray(A), b_ub=np.asarray(b), bounds=bounds, method="highs")
            if res.success:
                actions = np.clip(res.x[ia:ia+H], -1.0, 1.0)         # 최적 행동열
                nets = np.asarray(base_net) + actions*self.nominal   # 그때의 순전력열
                return actions, nets
        except Exception:
            pass
        return np.zeros(H), np.asarray(base_net, dtype=float)        # 실패 시 아무것도 안 함

## 3. Phase 1 — 분산 최적화 에이전트 (MPCAgent)

관측에서 각 건물의 부하·태양광·SoC·요금·탄소를 뽑아, 예측을 만들고, **건물마다 위 LP를 풀어** 행동을 결정합니다. 정전 대응이 없는 이 기본형만으로도 **중앙집중 MPC와 동일한 성능**(평상시 최적)을 냅니다.

In [ ]:
@dataclass
class MPCConfig:
    """MPC 하이퍼파라미터."""
    horizon: int = 8          # 몇 시간 앞을 보는가
    carbon_weight: float = 1.0 # 탄소 비중
    ramp_weight: float = 0.3   # 램핑 비중
    soc_min: float = 0.0; soc_max: float = 1.0

class MPCAgent:
    def __init__(self, env, config=None):
        self.env = env; self.cfg = config or MPCConfig(); self.n = len(env.buildings)
        on = env.observation_names[0]            # 관측 벡터의 각 칸 이름 목록
        # --- 관측 벡터에서 필요한 값들의 위치(인덱스)를 이름으로 찾아둔다 ---
        self._ix_load  = [i for i,nm in enumerate(on) if nm=="non_shiftable_load"]   # 건물별 부하
        self._ix_solar = [i for i,nm in enumerate(on) if nm=="solar_generation"]      # 건물별 태양광
        self._ix_soc   = [i for i,nm in enumerate(on) if nm=="electrical_storage_soc"] # 건물별 SoC
        self._ix_hour  = on.index("hour") if "hour" in on else None                   # 현재 시각
        self._ix_price = on.index("electricity_pricing") if "electricity_pricing" in on else None
        self._ix_carbon= on.index("carbon_intensity") if "carbon_intensity" in on else None
        self._ix_price_pred = [on.index(f"electricity_pricing_predicted_{k}") if f"electricity_pricing_predicted_{k}" in on else None for k in (1,2,3)]
        # --- 건물별 배터리 스펙으로 LP 컨트롤러를 하나씩 생성 ---
        self.capacity = np.array([b.electrical_storage.capacity for b in env.buildings], dtype=float)
        self.nominal  = np.array([float(getattr(b.electrical_storage,"nominal_power",5.0)) for b in env.buildings])
        self.controllers = [MPCBattery(self.capacity[b], self.nominal[b], horizon=self.cfg.horizon,
                            soc_min=self.cfg.soc_min, soc_max=self.cfg.soc_max,
                            carbon_weight=self.cfg.carbon_weight, ramp_weight=self.cfg.ramp_weight) for b in range(self.n)]
        self.fc = ForecasterMPC(self.n, self.cfg.horizon)   # 예측기 1개(모든 건물 공유)
        self._step = 0

    def predict(self, observations, deterministic=None):
        """환경이 주는 관측 -> 모든 건물의 행동 리스트 반환."""
        obs = np.asarray(observations[0], dtype=float)       # 중앙 에이전트라 관측은 1개로 합쳐져 옴
        hour = int(obs[self._ix_hour]) if self._ix_hour is not None else (self._step%24)+1
        load  = np.array([obs[i] for i in self._ix_load])    # 건물별 부하 벡터
        solar = np.array([obs[i] for i in self._ix_solar])   # 건물별 태양광
        soc   = np.array([obs[i] for i in self._ix_soc])     # 건물별 SoC
        price = float(obs[self._ix_price]) if self._ix_price is not None else 0.0
        carbon= float(obs[self._ix_carbon]) if self._ix_carbon is not None else 0.0
        price_pred = [float(obs[i]) for i in self._ix_price_pred if i is not None]
        self.fc.update(hour, load, solar, price, carbon, price_pred)   # 예측기에 이번 관측 반영
        price_fc = self.fc.price_forecast(hour); carbon_fc = self.fc.carbon_forecast(hour)
        actions = np.zeros(self.n)
        for b in range(self.n):                              # 건물마다 자기 LP를 풀어 행동 결정
            actions[b] = self.controllers[b].decide(float(soc[b]), self.fc.net_forecast(b,hour), price_fc, carbon_fc)
        self._step += 1
        return [np.clip(actions, -1.0, 1.0).astype(float).tolist()]   # [-1,1]로 잘라 반환

## 4. Phase 3 협력층 — 블랙보드·건물 에이전트·공유변수·역할 배정

여기서부터 "에이전틱 메시"입니다. 모든 에이전트는 **공유 블랙보드**에 정보를 올리고(post) 읽습니다.
- **건물 에이전트**: 자기 상태·예측을 올리고, 블랙보드 정보로 자기 MPC를 풉니다.
- **공유변수 에이전트**: 요금·탄소 예측을 모두에게 broadcast.
- **역할 배정**: 건물 상태에 따라 역할(여유 방전 가능/잉여 흡수/예비 보존/유연)을 라벨링 — *설명용이며 행동은 안 바꿉니다.*

In [ ]:
class Blackboard:
    """모든 에이전트가 읽고 쓰는 공유 게시판(통신 버스)."""
    def __init__(self):
        self.net_fc = {}        # 건물 -> 순부하 예측
        self.states = {}        # 건물 -> {soc, load, solar}
        self.price_fc = None; self.carbon_fc = None   # 요금·탄소 예측
        self.roles = {}         # 건물 -> 역할 라벨
        self.lead = {}          # 공유변수 대표 건물
        self.reserve_floor = 0.0  # 정전 감지 에이전트가 설정하는 reserve 하한
        self.outage_risk = 0.0

class BuildingAgent:
    """한 건물의 전문가: 자기 순부하를 올리고, 자기 배터리 MPC를 푼다."""
    def __init__(self, b, controller):
        self.b = b; self.controller = controller; self.last = {}
    def post(self, bb, soc, load, solar, net_fc):
        # 내 상태와 예측을 블랙보드에 올린다
        bb.states[self.b] = {"soc":float(soc), "load":float(load), "solar":float(solar)}
        bb.net_fc[self.b] = net_fc
    def act(self, bb):
        # 블랙보드의 (내 예측 + 요금·탄소 + reserve 하한)으로 내 LP를 풀어 행동 결정
        net = bb.net_fc[self.b]; soc = bb.states[self.b]["soc"]
        actions, _ = self.controller.solve(soc, net, bb.price_fc, bb.carbon_fc, reserve_floor=bb.reserve_floor)
        a0 = float(actions[0])
        self.last = {"action":a0, "soc":soc, "net0":float(net[0]), "reserve":bb.reserve_floor}  # 설명용 기록
        return a0
    def report(self, bb):
        # 사람이 읽을 수 있는 자연어 근거(설명가능성용)
        role = bb.roles.get(self.b, "self_expert"); a = self.last.get("action", 0.0)
        verb = "방전" if a < -1e-3 else ("충전" if a > 1e-3 else "유지")
        return {"agent":f"building_{self.b}", "role":role, "action":round(a,3), "soc":round(self.last.get("soc",0.0),3),
                "reason":f"{verb} (a={a:+.2f}, SoC {self.last.get('soc',0.0):.2f}, 예측 순부하 {self.last.get('net0',0.0):.1f}kWh)"}

class SharedVarAgent:
    """구역 공통 변수(요금 또는 탄소)를 예측해 블랙보드에 broadcast."""
    def __init__(self, name): self.name = name; self.last = None
    def post(self, bb, forecast):
        if self.name == "price": bb.price_fc = forecast      # 요금 예측 게시
        else: bb.carbon_fc = forecast                        # 탄소 예측 게시
        self.last = forecast
    def report(self, bb):
        f = self.last; nxt = [round(float(x),3) for x in (f[1:4] if f is not None and len(f) > 3 else [])]
        return {"agent":f"{self.name}_agent", "role":f"{self.name}_lead", "forecast_next3":nxt, "reason":f"{self.name} 다음 3시간 예측 {nxt}"}

class RoleAssigner:
    """매 step 건물 상태로 역할을 동적 배정 (라벨만, 행동은 불변)."""
    def assign(self, bb, soc_high=0.6, soc_low=0.3):
        roles = {}
        for b, st in bb.states.items():
            soc = st["soc"]; net0 = float(bb.net_fc[b][0]) if b in bb.net_fc else 0.0
            if   soc >= soc_high and net0 > 0: roles[b] = "relief_capable"   # 여유 SoC + 부하 → 방전으로 도울 수 있음
            elif net0 < 0 and soc < 0.9:       roles[b] = "absorber"          # 잉여(태양광) → 흡수
            elif soc <= soc_low:               roles[b] = "reserve_holder"    # 낮은 SoC → 보존
            else:                              roles[b] = "flexible"          # 그 외 → 유연
        bb.roles = roles
        if bb.states:   # 공유변수 대표는 SoC 중앙값 건물(가장 대표적)
            order = sorted(bb.states, key=lambda b: bb.states[b]["soc"]); mid = order[len(order)//2]
            bb.lead = {"price":mid, "carbon":mid}

## 5. Phase 2 핵심 — 정전 감지 에이전트 (선택적 reserve)

정전 대비의 두뇌입니다. 관측된 정전에서 **시간대별 정전 빈도**를 학습하고, 빈도가 가장 높은 시간대의 **50%(`risk_threshold`) 이상인 시간만 "위험"** 으로 판단합니다.

이렇게 *임계값*을 두는 이유: "한 번이라도 정전난 시간 = 위험"으로 하면, 긴 정전이 자정을 넘겨 번지면서 거의 모든 시간이 위험으로 찍혀 **reserve가 항상 켜지는**(SoC를 늘 높게 잡는) 문제가 생깁니다. 임계값을 쓰면 **진짜 위험 시간대(저녁)만** 잡아 reserve가 선택적으로 켜지고, 나머지 시간엔 평상시처럼 SoC가 cycling 합니다.

In [ ]:
class OutageRiskAgent:
    """정전을 예지해 reserve를 켜는 전문 에이전트 (관측만 사용 → causal)."""
    def __init__(self, horizon, max_reserve=0.4, anticipate=3, risk_threshold=0.5):
        self.H = horizon
        self.anticipate = int(min(anticipate, horizon))   # 위험 시간대 몇 시간 전부터 대비할지
        self.max_reserve = max_reserve                    # 최대 reserve 하한(SoC 비율)
        self.risk_threshold = float(risk_threshold)       # '위험' 판단 임계값(최고빈도 대비 비율)
        self.hour_total  = np.zeros(24)                   # 시각별 관측 횟수
        self.hour_outage = np.zeros(24)                   # 시각별 정전 누적(빈도 분자)

    def observe(self, hour, outage_fraction):
        """매 step: 지금 시각에 정전이 얼마나 났는지 누적."""
        h = int(hour) % 24
        self.hour_total[h]  += 1.0
        self.hour_outage[h] += float(outage_fraction)

    def _risky_hour(self, h):
        """이 시각이 위험한가? = 그 시각 정전빈도가 최고빈도의 risk_threshold배 이상인가."""
        freq = self.hour_outage / np.maximum(self.hour_total, 1.0)   # 시각별 정전빈도
        peak = float(freq.max())
        if peak <= 0.0: return False                                  # 아직 정전 관측 없음 → 위험 아님
        return float(freq[int(h) % 24]) >= self.risk_threshold * peak # 임계값 이상이면 위험

    def assess(self, hour):
        """(위험도, reserve 하한) 반환. 앞으로 anticipate시간 내 위험 시각이 있으면 reserve를 켠다."""
        risk = 1.0 if any(self._risky_hour(hour+k) for k in range(self.anticipate+1)) else 0.0
        return risk, float(risk) * self.max_reserve

    def report(self, hour, risk, reserve_floor):
        return {"agent":"outage_risk_agent", "role":"outage_detector", "risk":round(risk,3), "reserve_floor":round(reserve_floor,3),
                "reason":(f"다가오는 {self.anticipate}시간 내 정전 위험 학습 → reserve {reserve_floor:.2f} 활성" if reserve_floor>1e-3 else "정전 위험 낮음 → reserve 0 (= 평상시 MPC)")}

## 6. 전체 오케스트레이션 (MASMPCAgent)

매 step의 흐름을 한 곳에 모읍니다:
1. **정전 감지** → reserve 하한 설정 (Phase 2)
2. **예측 게시** (건물·공유변수 에이전트) (Phase 1·3)
3. **역할 배정** (Phase 3)
4. **건물별 MPC** 풀어 행동 결정 (Phase 1, reserve 반영)
5. **긴급 방전 override**: 지금 정전 중인 건물은 LP 해를 덮어쓰고 부하만큼 방전 (Phase 2)

In [ ]:
class MASMPCAgent(MPCAgent):
    """에이전트 소통 → 건물별 MPC. 정전 없으면 MPCAgent와 동일, 정전 시 reserve+긴급방전으로 회복력 확보."""
    def __init__(self, env, config=None, enable_reserve=True, max_reserve=0.4):
        super().__init__(env, config)                                            # Phase 1 초기화 재사용
        self.building_agents = [BuildingAgent(b, self.controllers[b]) for b in range(self.n)]  # 건물 에이전트들
        self.price_agent  = SharedVarAgent("price")                              # 요금 전담
        self.carbon_agent = SharedVarAgent("carbon")                             # 탄소 전담
        self.assigner = RoleAssigner()                                           # 역할 배정자
        self.enable_reserve = enable_reserve                                     # 정전 대응 on/off
        self.risk_agent = OutageRiskAgent(self.cfg.horizon, max_reserve=max_reserve)  # 정전 감지(선택적 reserve)
        self.reports = []                                                        # step별 설명 기록

    def predict(self, observations, deterministic=None):
        # --- 관측 파싱 (MPCAgent와 동일) ---
        obs = np.asarray(observations[0], dtype=float)
        hour = int(obs[self._ix_hour]) if self._ix_hour is not None else (self._step%24)+1
        load = np.array([obs[i] for i in self._ix_load]); solar = np.array([obs[i] for i in self._ix_solar])
        soc = np.array([obs[i] for i in self._ix_soc])
        price = float(obs[self._ix_price]) if self._ix_price is not None else 0.0
        carbon= float(obs[self._ix_carbon]) if self._ix_carbon is not None else 0.0
        price_pred = [float(obs[i]) for i in self._ix_price_pred if i is not None]
        self.fc.update(hour, load, solar, price, carbon, price_pred)

        bb = Blackboard()                                  # 이번 step 블랙보드
        risk = reserve_floor = 0.0; outage_now = [False]*self.n
        # --- (1) 정전 감지: 지금 정전 관측 → 학습 → reserve 하한 결정 ---
        if self.enable_reserve:
            outage_now = [bool(getattr(b, "power_outage", False)) for b in self.env.buildings]  # 건물별 현재 정전 여부
            self.risk_agent.observe(hour, float(np.mean([1.0 if o else 0.0 for o in outage_now])))  # 학습
            risk, reserve_floor = self.risk_agent.assess(hour)                                   # 위험·reserve
        bb.reserve_floor = reserve_floor; bb.outage_risk = risk

        # --- (2) 예측 게시: 공유변수·건물 에이전트가 블랙보드에 올림 ---
        self.price_agent.post(bb, self.fc.price_forecast(hour))
        self.carbon_agent.post(bb, self.fc.carbon_forecast(hour))
        for b in range(self.n):
            self.building_agents[b].post(bb, float(soc[b]), float(load[b]), float(solar[b]), self.fc.net_forecast(b, hour))
        # --- (3) 역할 동적 배정 ---
        self.assigner.assign(bb)
        # --- (4) 건물별 MPC 풀어 행동 결정 (reserve 하한 반영) ---
        actions = np.array([self.building_agents[b].act(bb) for b in range(self.n)])

        # --- (5) 긴급 방전: 지금 정전 중인(섬 고립) 건물은 LP를 덮어쓰고 부하만큼 방전 ---
        n_deploy = 0
        for b in range(self.n):
            if outage_now[b]:                              # 이 건물이 지금 정전이면
                nominal = max(self.controllers[b].nominal, 1e-6)
                need = float(load[b] - solar[b])           # 채워야 할 순부하(부하-태양광)
                actions[b] = float(np.clip(-need/nominal, -1.0, 1.0))   # 그만큼 방전(음수 행동)
                self.building_agents[b].last["action"] = actions[b]; n_deploy += 1

        # --- 설명 기록(나중에 LLM/프론트엔드에서 사용) ---
        step_reports = [self.risk_agent.report(hour, risk, reserve_floor)]
        step_reports += [self.building_agents[b].report(bb) for b in range(self.n)]
        self.reports.append({"step":self._step, "hour":hour, "reports":step_reports,
                             "outage_risk":round(risk,3), "reserve_floor":round(reserve_floor,3), "emergency_deploy":n_deploy})
        self._step += 1
        return [np.clip(actions, -1.0, 1.0).astype(float).tolist()]

## 7. 정전 시나리오 주입

CityLearn 2022는 정전이 꺼져 있습니다. 저녁 피크(17~20시)에 확률적 정전을 켜고, **모든 컨트롤러에 같은 시드**를 줘서 동일한 정전을 겪게 합니다(공정 비교).

In [ ]:
def enable_outages(env, saifi=80.0, caidi=180.0, start_hours=(17,18,19,20), seed=0):
    """모든 건물에 저녁 피크 정전을 주입. saifi=연 정전 횟수, caidi=평균 지속(분)."""
    from citylearn.power_outage import ReliabilityMetricsPowerOutage
    start_steps = [int(h) for h in start_hours]                 # 정전 시작 가능 시각
    for i, b in enumerate(env.buildings):
        b.simulate_power_outage = True; b.stochastic_power_outage = True
        m = ReliabilityMetricsPowerOutage(saifi=saifi, caidi=caidi, start_time_steps=start_steps)
        m.random_seed = seed + i                                # 같은 seed → 컨트롤러 간 동일 정전(공정)
        b.stochastic_power_outage_model = m
    return env

## 8. 실행 + 비교 — 무제어 / 규칙기반 / 강화학습 / MPC / 우리 모델

정전 회복력(미공급 에너지 `unserved`, 낮을수록 좋음)과 표준 점수를 비교합니다. `FULL=True`면 1년, 기본은 60일(빠른 확인).

In [ ]:
from citylearn.citylearn import CityLearnEnv
from citylearn.agents.base import BaselineAgent     # 무제어(아무 것도 안 함)
from citylearn.agents.rbc import BasicBatteryRBC    # 규칙기반 배터리

FULL = False                                        # True면 1년(8760 step), False면 60일
END  = None if FULL else 24*60-1
OUT  = dict(saifi=80.0, caidi=180.0, start_hours=(17,18,19,20), seed=0)
def make_env(central=True):
    kw = dict(central_agent=central)                # 중앙 에이전트 여부(SAC만 분산)
    if END is not None: kw.update(simulation_start_time_step=0, simulation_end_time_step=END)
    return enable_outages(CityLearnEnv('citylearn_challenge_2022_phase_all', **kw), **OUT)
def run(make, fac):                                 # 환경 생성 -> 에이전트 -> 끝까지 시뮬레이션
    e = make(); a = fac(e); o,_ = e.reset()
    while not e.terminated:
        act = a.predict(o); o,_,_,t,tr = e.step(act)
        if t or tr: break
    return e
KPI4 = ['cost_total','carbon_emissions_total','ramping_average','daily_one_minus_load_factor_average']
def metrics(e):                                     # (표준점수, 회복력) 계산
    df = e.evaluate(); has = ('name' in df.columns) and (df['name'].astype(str)=='District').any(); d = {}
    for _, r in df.iterrows():
        cf = str(r.get('cost_function'))
        if cf in KPI4 + ['power_outage_normalized_unserved_energy_total']:
            if has and str(r.get('name',''))!='District': continue
            d[cf] = r['value']
    return float(np.nanmean([d.get(k) for k in KPI4])), float(d.get('power_outage_normalized_unserved_energy_total', np.nan))

# --- 강화학습(SAC): 학습된 가중치를 불러와 실행 (번들 있을 때만) ---
def load_sac(env, path):
    import torch
    from citylearn.agents.sac import SACRBC
    bun = torch.load(path, map_location='cpu', weights_only=False)
    ag = SACRBC(env, **dict(bun['agent_kwargs']))
    for i, sd in enumerate(bun['policy_net_state_dicts']):     # 건물별 정책 가중치 로드
        ag.policy_net[i].load_state_dict(sd); ag.policy_net[i].eval()
    ag.norm_mean = bun['norm_mean']; ag.norm_std = bun['norm_std']; ag.normalized = bun['normalized']
    ag.end_exploration_time_step = -1                         # 탐험 끄고 학습된 정책만 사용
    return ag
class SACWrap:
    def __init__(self, env, path): self.a = load_sac(env, path)
    def predict(self, o, deterministic=True): return self.a.predict(o, deterministic=True)

# --- 4개 중앙 컨트롤러 실행 ---
rows = []
for name, fac in [('무제어', lambda e: BaselineAgent(e)),
                  ('규칙기반(RBC)', lambda e: BasicBatteryRBC(e)),
                  ('MPC(원논문)', lambda e: MPCAgent(e, MPCConfig())),
                  ('우리 모델(Mesh)', lambda e: MASMPCAgent(e, MPCConfig()))]:
    e = run(lambda: make_env(True), fac); sc, res = metrics(e)
    rows.append({'controller':name, '표준점수(↓)':round(sc,4), '회복력 unserved(↓)':round(res,4)})
    print(f"  {name:16s} score {sc:.4f} | unserved {res:.4f}")
# --- 강화학습(SAC): 분산 환경 + 번들 (없으면 자동 스킵) ---
try:
    e = run(lambda: make_env(False), lambda e: SACWrap(e, str(SAC_BUNDLE))); sc, res = metrics(e)
    rows.insert(2, {'controller':'강화학습(SAC)', '표준점수(↓)':round(sc,4), '회복력 unserved(↓)':round(res,4)})
    print(f"  강화학습(SAC)     score {sc:.4f} | unserved {res:.4f}")
except Exception as ex:
    print("  강화학습(SAC) 스킵:", type(ex).__name__, ex)
display(pd.DataFrame(rows))

## 9. 우리 모델 동작 확인 — 선택적 reserve / SoC cycling / 역할 분배

"매 step SoC를 그냥 높게 잡은 것 아니냐"는 의문을 직접 확인합니다: 위험 시간대가 **저녁만**으로 좁게 잡히고, reserve가 **일부 시간만**(≈30%) 켜지며, SoC는 **정상 cycling**(std>0)하는지 봅니다.

In [ ]:
e = make_env(True); a = MASMPCAgent(e, MPCConfig()); o,_ = e.reset()   # 우리 모델 1회 실행
while not e.terminated:
    act = a.predict(o); o,_,_,t,tr = e.step(act)
    if t or tr: break

freq = a.risk_agent.hour_outage / np.maximum(a.risk_agent.hour_total, 1)   # 학습된 시간대별 정전빈도
thr  = 0.5 * freq.max()                                                     # 위험 임계값
risky = [h for h in range(24) if freq[h] >= thr and freq[h] > 0]           # 위험으로 판단된 시간대
rf = np.array([r['reserve_floor'] for r in a.reports])                      # step별 reserve 하한
print("학습된 위험 시간대(선택적):", risky)
print(f"reserve 활성 비율: {100*np.mean(rf>0):.0f}%   (선택적이라 100%가 아님)")

soc1 = np.asarray(e.buildings[1].electrical_storage.soc, dtype=float)       # 건물1 SoC 궤적
print(f"건물1 SoC: 평균 {soc1.mean():.2f}  std {soc1.std():.2f}  min {soc1.min():.2f}  max {soc1.max():.2f}   (std>0 → 충·방전 cycling)")

# --- 역할 분배 스냅샷: 역할이 다양한 저녁 한 step ---
snap = [r for r in a.reports if r['hour'] in (18,19,20)]
rp = max(snap, key=lambda r: len(set(x.get('role') for x in r['reports'] if x['agent'].startswith('building'))))
ko = {'relief_capable':'여유 방전 가능','absorber':'잉여 흡수','reserve_holder':'예비 보존','flexible':'유연'}
print(f"\n역할 분배 스냅샷 (step {rp['step']}, {rp['hour']}시):")
seen = {}
for r in rp['reports']:
    if r['agent'].startswith('building') and r['role'] not in seen:
        seen[r['role']] = r
        print(f"  건물 {r['agent'].split('_')[1]:>2}: [{ko.get(r['role'],r['role'])}]  SoC {r['soc']*100:.0f}%")

## 10. 요약

- **Phase 1 (분산 최적화)**: 각 건물이 자기 MPC를 풀어 평상시 중앙 MPC와 동일 성능.
- **Phase 2 (돌발 대응)**: 정전 위험을 *선택적으로* 학습(저녁 위험창만) → reserve 선축적 + 정전 순간 긴급 방전. SoC는 평상시 정상 cycling.
- **Phase 3 (역할 분배)**: 건물 상태로 역할 동적 배정(설명가능성 기반).
- **결과**: 우리 모델이 정전 회복력(unserved)에서 무제어·RBC·SAC·MPC를 모두 앞선다.

> 모든 코드가 이 노트북 안에 있으므로, 위에서부터 차례로 실행하면 전체를 재현할 수 있습니다.